# Semana 3: Regresión Lineal y Regularización
## Notebook Conceptual (NB1) – Experimentación con Datos Dummy

**Propósito:** Comprender la regresión lineal desde sus fundamentos matemáticos, implementarla manualmente, y luego usar scikit-learn explorando el efecto de la regularización.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Implementar la ecuación normal para regresión lineal.
- Calcular el gradiente del MSE y dar pasos de gradiente descendente.
- Comparar coeficientes con LinearRegression de scikit-learn.
- Experimentar con regularización Ridge y Lasso.
- Visualizar funciones de coste con y sin regularización.
- Medir tiempos de ejecución y entender complejidad computacional.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y fijamos la semilla para reproducibilidad.

In [ ]:
# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import time

# Scikit-learn
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# Configuración de visualización
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Semilla
np.random.seed(42)

print("Librerías importadas correctamente.")

---
## 1. Generación de Datos Sintéticos Univariados

Creamos datos con una relación lineal más ruido gaussiano para visualizar fácilmente el ajuste.

In [ ]:
# Parámetros verdaderos
true_b0 = 4.5  # intercepto
true_b1 = 2.8  # pendiente

# Generamos datos
n_samples = 100
X = np.linspace(0, 10, n_samples).reshape(-1, 1)
y = true_b0 + true_b1 * X.ravel() + np.random.randn(n_samples) * 3  # añadimos ruido

# Visualizamos
plt.scatter(X, y, alpha=0.6, label='Datos')
plt.plot(X, true_b0 + true_b1 * X, 'r--', label=f'Recta real: y = {true_b0} + {true_b1}X')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Datos sintéticos para regresión lineal')
plt.legend()
plt.grid(True)
plt.show()

---
## 2. Implementación de la Ecuación Normal

La ecuación normal proporciona una solución analítica para los coeficientes de regresión lineal:

$$\hat{\beta} = (X^T X)^{-1} X^T y$$

Recordemos que debemos incluir una columna de unos para el intercepto.

In [ ]:
# Añadimos columna de unos para el intercepto
X_b = np.c_[np.ones((n_samples, 1)), X]  # X_b = [1, X]

# Calculamos con ecuación normal
beta_normal = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y

print(f"Coeficientes por ecuación normal:")
print(f"Intercepto (β0): {beta_normal[0]:.4f}")
print(f"Pendiente (β1): {beta_normal[1]:.4f}")
print(f"Valores reales: β0={true_b0}, β1={true_b1}")

### 2.1. Cálculo del MSE y visualización de la recta ajustada

In [ ]:
# Predicciones
y_pred_normal = X_b @ beta_normal

# MSE
mse_normal = mean_squared_error(y, y_pred_normal)
print(f"MSE (ecuación normal): {mse_normal:.4f}")

# Visualización
plt.scatter(X, y, alpha=0.6, label='Datos')
plt.plot(X, y_pred_normal, 'b-', linewidth=2, label=f'Recta ajustada: y = {beta_normal[0]:.2f} + {beta_normal[1]:.2f}X')
plt.plot(X, true_b0 + true_b1 * X, 'r--', label='Recta real')
plt.xlabel('X')
plt.ylabel('y')
plt.title(f'Ajuste por ecuación normal (MSE = {mse_normal:.2f})')
plt.legend()
plt.grid(True)
plt.show()

---
## 3. Gradiente Descendente Manual

La función de coste MSE es:
$$J(\beta) = \frac{1}{n} \sum_{i=1}^n (y_i - \hat{y}_i)^2$$

El gradiente con respecto a β es:
$$\nabla J(\beta) = -\frac{2}{n} X^T (y - X\beta)$$

Implementamos el gradiente descendente paso a paso.

In [ ]:
# Inicializamos coeficientes aleatorios
beta_gd = np.random.randn(2)

# Hiperparámetros
learning_rate = 0.01
n_iterations = 1000
m = n_samples

# Guardamos el historial de costes y coeficientes
cost_history = []
beta_history = [beta_gd.copy()]

for iteration in range(n_iterations):
    # Predicciones
    y_pred = X_b @ beta_gd
    
    # Cálculo del coste
    cost = (1/m) * np.sum((y - y_pred)**2)
    cost_history.append(cost)
    
    # Gradiente
    gradient = -(2/m) * X_b.T @ (y - y_pred)
    
    # Actualización
    beta_gd = beta_gd - learning_rate * gradient
    beta_history.append(beta_gd.copy())

beta_history = np.array(beta_history)

print(f"Coeficientes por gradiente descendente:")
print(f"Intercepto (β0): {beta_gd[0]:.4f}")
print(f"Pendiente (β1): {beta_gd[1]:.4f}")
print(f"Comparación ecuación normal: β0={beta_normal[0]:.4f}, β1={beta_normal[1]:.4f}")

In [ ]:
# Visualización de la convergencia
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Evolución del coste
axes[0].plot(cost_history)
axes[0].set_xlabel('Iteración')
axes[0].set_ylabel('MSE')
axes[0].set_title('Evolución del coste durante GD')
axes[0].grid(True)

# Trayectoria de los coeficientes (solo últimos 100 para claridad)
axes[1].plot(beta_history[-100:, 0], beta_history[-100:, 1], 'o-', markersize=3)
axes[1].scatter(beta_normal[0], beta_normal[1], color='red', s=100, marker='*', label='Ecuación normal')
axes[1].set_xlabel('β0 (intercepto)')
axes[1].set_ylabel('β1 (pendiente)')
axes[1].set_title('Trayectoria de coeficientes (últimas 100 iteraciones)')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

---
## 4. Comparación con LinearRegression de scikit-learn

Usamos la implementación optimizada de scikit-learn para verificar nuestros resultados.

In [ ]:
# Entrenamos modelo
model_sk = LinearRegression()
model_sk.fit(X, y)

# Coeficientes
print(f"LinearRegression scikit-learn:")
print(f"Intercepto: {model_sk.intercept_:.4f}")
print(f"Coeficiente: {model_sk.coef_[0]:.4f}")
print(f"\nNuestra ecuación normal:")
print(f"Intercepto: {beta_normal[0]:.4f}")
print(f"Coeficiente: {beta_normal[1]:.4f}")
print(f"\nNuestro gradiente descendente:")
print(f"Intercepto: {beta_gd[0]:.4f}")
print(f"Coeficiente: {beta_gd[1]:.4f}")

---
## 5. Regularización Ridge y Lasso

Ahora exploramos cómo afecta la regularización a los coeficientes. Generamos datos multivariados con más variables para observar el efecto.

In [ ]:
# Generamos datos con 10 variables
np.random.seed(42)
n_samples_multi = 200
n_features = 10
X_multi = np.random.randn(n_samples_multi, n_features)

# Coeficientes verdaderos (solo algunos distintos de cero)
true_coef = np.array([3, 1.5, 0, 0, 2, 0, 0, 0, -1, 0.5])
y_multi = X_multi @ true_coef + np.random.randn(n_samples_multi) * 0.5

# Dividimos en train/test
X_train, X_test, y_train, y_test = train_test_split(X_multi, y_multi, test_size=0.2, random_state=42)

# Estandarizamos (importante para regularización)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### 5.1. Ridge (L2): variando alpha y observando coeficientes

In [ ]:
alphas = np.logspace(-3, 3, 50)
coefs_ridge = []

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_scaled, y_train)
    coefs_ridge.append(ridge.coef_)

coefs_ridge = np.array(coefs_ridge)

plt.figure(figsize=(10, 6))
for i in range(n_features):
    plt.plot(alphas, coefs_ridge[:, i], label=f'X{i+1}')
plt.xscale('log')
plt.xlabel('alpha (escala log)')
plt.ylabel('Coeficientes')
plt.title('Trayectoria de regularización - Ridge')
plt.legend(loc='best', ncol=2)
plt.grid(True)
plt.show()

### 5.2. Lasso (L1): selección de variables

In [ ]:
coefs_lasso = []

for alpha in alphas:
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_train_scaled, y_train)
    coefs_lasso.append(lasso.coef_)

coefs_lasso = np.array(coefs_lasso)

plt.figure(figsize=(10, 6))
for i in range(n_features):
    plt.plot(alphas, coefs_lasso[:, i], label=f'X{i+1}')
plt.xscale('log')
plt.xlabel('alpha (escala log)')
plt.ylabel('Coeficientes')
plt.title('Trayectoria de regularización - Lasso')
plt.legend(loc='best', ncol=2)
plt.grid(True)
plt.show()

### 5.3. Comparación de coeficientes para un alpha específico

In [ ]:
alpha_ejemplo = 0.1

ridge_ejemplo = Ridge(alpha=alpha_ejemplo)
ridge_ejemplo.fit(X_train_scaled, y_train)

lasso_ejemplo = Lasso(alpha=alpha_ejemplo, max_iter=10000)
lasso_ejemplo.fit(X_train_scaled, y_train)

print("Coeficientes verdaderos:", true_coef)
print("\nRidge (L2):")
print(ridge_ejemplo.coef_.round(3))
print("\nLasso (L1):")
print(lasso_ejemplo.coef_.round(3))

# Número de ceros en Lasso
print(f"\nLasso ha hecho {np.sum(np.abs(lasso_ejemplo.coef_) < 1e-6)} coeficientes exactamente cero")

---
## 6. Visualización de la Función de Coste (2D) con y sin Regularización

Para visualizar el efecto de la regularización, usamos un modelo con solo dos coeficientes (sin contar intercepto) y representamos el contorno de la función de coste.

In [ ]:
# Generamos datos con 2 variables
np.random.seed(42)
X_2d = np.random.randn(100, 2)
true_coef_2d = np.array([2.5, -1.5])
y_2d = X_2d @ true_coef_2d + np.random.randn(100) * 0.5

# Creamos una malla de posibles coeficientes
b1_range = np.linspace(-5, 5, 50)
b2_range = np.linspace(-5, 5, 50)
B1, B2 = np.meshgrid(b1_range, b2_range)

# Calculamos MSE para cada combinación
mse_values = np.zeros_like(B1)
ridge_values = np.zeros_like(B1)
lasso_values = np.zeros_like(B1)

lambda_reg = 1.0  # hiperparámetro de regularización

for i in range(len(b1_range)):
    for j in range(len(b2_range)):
        beta = np.array([B1[j, i], B2[j, i]])
        y_pred = X_2d @ beta
        mse = np.mean((y_2d - y_pred)**2)
        mse_values[j, i] = mse
        ridge_values[j, i] = mse + lambda_reg * np.sum(beta**2)  # MSE + L2
        lasso_values[j, i] = mse + lambda_reg * np.sum(np.abs(beta))  # MSE + L1

# Visualización
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

contour1 = axes[0].contourf(B1, B2, mse_values, levels=50, cmap='viridis')
axes[0].scatter(true_coef_2d[0], true_coef_2d[1], color='red', marker='*', s=200, label='Coef. verdaderos')
axes[0].set_xlabel('β1')
axes[0].set_ylabel('β2')
axes[0].set_title('MSE sin regularización')
plt.colorbar(contour1, ax=axes[0])

contour2 = axes[1].contourf(B1, B2, ridge_values, levels=50, cmap='plasma')
axes[1].scatter(true_coef_2d[0], true_coef_2d[1], color='red', marker='*', s=200, label='Coef. verdaderos')
axes[1].set_xlabel('β1')
axes[1].set_ylabel('β2')
axes[1].set_title(f'MSE + Ridge (L2, λ={lambda_reg})')
plt.colorbar(contour2, ax=axes[1])

contour3 = axes[2].contourf(B1, B2, lasso_values, levels=50, cmap='inferno')
axes[2].scatter(true_coef_2d[0], true_coef_2d[1], color='red', marker='*', s=200, label='Coef. verdaderos')
axes[2].set_xlabel('β1')
axes[2].set_ylabel('β2')
axes[2].set_title(f'MSE + Lasso (L1, λ={lambda_reg})')
plt.colorbar(contour3, ax=axes[2])

plt.tight_layout()
plt.show()

---
## 7. Medición de Tiempos: Ecuación Normal vs Gradiente Descendente

Comparamos la complejidad computacional variando el tamaño del dataset.

In [ ]:
def medir_tiempos(n_samples_list, n_features=5):
    tiempos_normal = []
    tiempos_gd = []
    
    for n in n_samples_list:
        # Generamos datos
        X_temp = np.random.randn(n, n_features)
        X_temp_b = np.c_[np.ones((n, 1)), X_temp]
        y_temp = X_temp @ np.random.randn(n_features) + np.random.randn(n)
        
        # Ecuación normal
        start = time.time()
        beta_norm = np.linalg.inv(X_temp_b.T @ X_temp_b) @ X_temp_b.T @ y_temp
        end = time.time()
        tiempos_normal.append(end - start)
        
        # Gradiente descendente (100 iteraciones fijas)
        beta_gd_temp = np.random.randn(n_features + 1)
        learning_rate_temp = 0.01
        start = time.time()
        for _ in range(100):
            y_pred = X_temp_b @ beta_gd_temp
            gradient = -(2/n) * X_temp_b.T @ (y_temp - y_pred)
            beta_gd_temp = beta_gd_temp - learning_rate_temp * gradient
        end = time.time()
        tiempos_gd.append(end - start)
    
    return tiempos_normal, tiempos_gd

# Probamos con diferentes tamaños
n_samples_list = [100, 500, 1000, 5000, 10000]
tiempos_normal, tiempos_gd = medir_tiempos(n_samples_list)

plt.figure(figsize=(10, 6))
plt.plot(n_samples_list, tiempos_normal, 'o-', label='Ecuación Normal (O(n³))')
plt.plot(n_samples_list, tiempos_gd, 's-', label='Gradiente Descendente (100 iter)')
plt.xlabel('Número de muestras')
plt.ylabel('Tiempo de ejecución (segundos)')
plt.title('Comparación de tiempos: Ecuación Normal vs Gradiente Descendente')
plt.legend()
plt.grid(True)
plt.show()

print("Tiempos (segundos):")
for i, n in enumerate(n_samples_list):
    print(f"n={n}: Normal={tiempos_normal[i]:.4f}s, GD={tiempos_gd[i]:.4f}s")

---
## 8. Experimentación Adicional: Efecto de α en Ridge y Lasso

Visualizamos cómo cambia el error en train y test al variar α.

In [ ]:
alphas_exp = np.logspace(-3, 2, 30)
train_errors_ridge = []
test_errors_ridge = []
train_errors_lasso = []
test_errors_lasso = []

for alpha in alphas_exp:
    # Ridge
    ridge_temp = Ridge(alpha=alpha)
    ridge_temp.fit(X_train_scaled, y_train)
    train_errors_ridge.append(mean_squared_error(y_train, ridge_temp.predict(X_train_scaled)))
    test_errors_ridge.append(mean_squared_error(y_test, ridge_temp.predict(X_test_scaled)))
    
    # Lasso
    lasso_temp = Lasso(alpha=alpha, max_iter=10000)
    lasso_temp.fit(X_train_scaled, y_train)
    train_errors_lasso.append(mean_squared_error(y_train, lasso_temp.predict(X_train_scaled)))
    test_errors_lasso.append(mean_squared_error(y_test, lasso_temp.predict(X_test_scaled)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(alphas_exp, train_errors_ridge, 'b-', label='Train')
axes[0].plot(alphas_exp, test_errors_ridge, 'r-', label='Test')
axes[0].set_xscale('log')
axes[0].set_xlabel('alpha')
axes[0].set_ylabel('MSE')
axes[0].set_title('Ridge: Error vs alpha')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(alphas_exp, train_errors_lasso, 'b-', label='Train')
axes[1].plot(alphas_exp, test_errors_lasso, 'r-', label='Test')
axes[1].set_xscale('log')
axes[1].set_xlabel('alpha')
axes[1].set_ylabel('MSE')
axes[1].set_title('Lasso: Error vs alpha')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

---
## 9. Conclusiones

Hemos explorado los fundamentos de la regresión lineal y la regularización:

✔️ **Ecuación normal**: Solución analítica directa, coste computacional O(n³).
✔️ **Gradiente descendente**: Método iterativo, escalable a grandes datasets.
✔️ **Comparación con scikit-learn**: Nuestras implementaciones manuales coinciden con las librerías estándar.
✔️ **Regularización Ridge (L2)**: Encoge coeficientes sin hacerlos cero, útil para multicolinealidad.
✔️ **Regularización Lasso (L1)**: Produce modelos dispersos (selección de variables).
✔️ **Visualización de funciones de coste**: La regularización modifica la superficie de coste, empujando coeficientes hacia cero.
✔️ **Complejidad computacional**: La ecuación normal se vuelve lenta para muchas muestras; el gradiente descendente escala mejor.

Estos conceptos son fundamentales y se extienden a modelos más complejos como regresión logística, redes neuronales, etc.

---
**Fin del Notebook Conceptual - Semana 3**